In [1]:
!pip install selenium requests pandas Pillow beautifulsoup4 lxml

In [2]:
import time, re, requests, pandas as pd, os
from PIL import Image
from io import BytesIO
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import shutil

In [3]:
options = Options()
options.binary_location = "/usr/bin/chromium"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1280,900")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_argument("user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
chromedriver_path = shutil.which("chromedriver") or "/usr/lib/chromium/chromedriver"
driver = webdriver.Chrome(service=Service(chromedriver_path), options=options)
print("Driver ready ✅")

Driver ready ✅


In [4]:
driver.get("https://www.houzz.com/photos/bathroom-ideas-phbr0-bp~t_712")
WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, "img[src]")))
time.sleep(4)
print("Page loaded ✅  Title:", driver.title)

Page loaded ✅  Title: 75 Bathroom Ideas You'll Love - April, 2026 | Houzz


In [5]:
def collect_page(driver, room_type):
    for scroll in range(0, 6000, 600):
        driver.execute_script(f"window.scrollTo(0, {scroll});")
        time.sleep(0.3)
    time.sleep(2)

    soup    = BeautifulSoup(driver.page_source, "lxml")
    results = []
    seen    = set()

    SKIP = ("save","view","add to","sign in","join","explore","browse",
            "see more","load more","similar","get","find","discover",
            "shop","click","learn","read")

    ROOM_KEYS = {"floor","wall","ceiling","kitchen","bedroom","bathroom",
                 "living","dining","office","cabinet","countertop","island",
                 "design","photo","example","style","space","room","layout",
                 "storage","renovation","remodel","transitional","modern",
                 "traditional","contemporary","farmhouse","light","window"}

    for img in soup.find_all("img"):
        src = img.get("src") or img.get("data-src") or ""
        if not src.startswith("http"): continue

        m = re.search(r"-w(\d+)-", src)
        if not m or int(m.group(1)) < 300: continue
        if any(s in src for s in ["logo","icon","avatar","profile"]): continue
        if src in seen: continue

        caption = None
        node = img

        for _ in range(6):
            node = node.parent
            if node is None: break

            for child in node.find_all(["p","span","div","a"], recursive=False):
                text = child.get_text(separator=" ", strip=True)
                words = text.split()

                if len(words) < 10: continue
                if text.lower().startswith(SKIP): continue

                lower_count = sum(1 for w in words if w.islower())
                if lower_count < 5: continue

                if not set(text.lower().split()) & ROOM_KEYS: continue

                caption = text
                break

            if caption: break

        if not caption: continue

        title = (img.get("alt") or "").strip() or " ".join(caption.split()[:6])
        seen.add(src)
        results.append({"title":title,"description":caption,
                        "image_url":src,"room_type":room_type})
    return results

In [6]:
page_data = collect_page(driver, "bathroom")
print(f"Page 1: {len(page_data)} good captions")
for item in page_data[:5]:
    nw = len(item["description"].split())
    print(f"  [{nw} words] {item['description'][:85]}")

Page 1: 40 good captions
  [28 words] Bathroom - country white tile white floor bathroom idea in Chicago with recessed-pane
  [57 words] We transformed a mid-terrace Victorian bathroom into a luxurious retreat by completel
  [35 words] Inspiration for a transitional multicolored tile mosaic tile floor, white floor and s
  [20 words] Monitor's Rest | Park City, Utah | CLB Architects Inspiration for a modern double-sin
  [849 words] Bathroom Ideas A bathroom remodel can make a huge impact on your homes comfort level,


In [7]:
URLS  = ['https://www.houzz.com/photos/bathroom-ideas-phbr0-bp~t_712', 'https://www.houzz.com/photos/modern-bathroom-ideas-phbr1-bp~t_712~s_2206', 'https://www.houzz.com/photos/traditional-bathroom-ideas-phbr1-bp~t_712~s_2107', 'https://www.houzz.com/photos/transitional-bathroom-ideas-phbr1-bp~t_712~s_2209', 'https://www.houzz.com/photos/farmhouse-bathroom-ideas-phbr1-bp~t_712~s_2213', 'https://www.houzz.com/photos/contemporary-bathroom-ideas-phbr1-bp~t_712~s_2207']
PAGES = 400
TARGET = 4000
all_data = []
all_seen = set()
print("Room:", "bathroom", "| Target:", TARGET)

for ui, base_url in enumerate(URLS):
    style = base_url.split("~s_")[-1] if "~s_" in base_url else "main"
    print(f"\n  URL {ui+1}/{len(URLS)} — {style}")
    empty = 0
    for page in range(1, PAGES+1):
        if len(all_data) >= TARGET:
            print("  ✅ Target reached"); break
        driver.get(f"{base_url}?pg={page}")
        print(f"    Page {page:03d}", end=" → ")
        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "img[src]")))
        except:
            print("TIMEOUT"); empty += 1
            if empty >= 3: print("  next URL"); break
            continue
        new = [i for i in collect_page(driver, "bathroom") if i["image_url"] not in all_seen]
        for i in new: all_seen.add(i["image_url"])
        all_data.extend(new)
        print(f"{len(new):3d} new | total: {len(all_data)}")
        empty = 0 if new else empty + 1
        if empty >= 5: print("  No new — next URL"); break
        time.sleep(1.5)
    if len(all_data) >= TARGET: break

print(f"\n✅ Done. Total: {len(all_data)} images")

Room: bathroom | Target: 4000

  URL 1/6 — main
    Page 001 →  40 new | total: 40
    Page 002 →  40 new | total: 80
    Page 003 →  20 new | total: 100
    Page 004 →  20 new | total: 120
    Page 005 →  20 new | total: 140
    Page 006 →  20 new | total: 160
    Page 007 →  20 new | total: 180
    Page 008 →  20 new | total: 200
    Page 009 →  20 new | total: 220
    Page 010 →  20 new | total: 240
    Page 011 →  21 new | total: 261
    Page 012 →  35 new | total: 296
    Page 013 →  20 new | total: 316
    Page 014 →  20 new | total: 336
    Page 015 →  20 new | total: 356
    Page 016 →  20 new | total: 376
    Page 017 →  20 new | total: 396
    Page 018 →  20 new | total: 416
    Page 019 →  20 new | total: 436
    Page 020 →  34 new | total: 470
    Page 021 →  20 new | total: 490
    Page 022 →  20 new | total: 510
    Page 023 →  20 new | total: 530
    Page 024 →  29 new | total: 559
    Page 025 →  28 new | total: 587
    Page 026 →  20 new | total: 607
    Page 027 →  20

In [8]:
driver.quit()
print('Driver closed ✅')

Driver closed ✅


In [9]:
df = pd.DataFrame(all_data).drop_duplicates(subset=["image_url"]).reset_index(drop=True)
df["wc"] = df["description"].str.split().str.len()
print(f"Total: {len(df)} | Min words: {df['wc'].min()} | Avg: {df['wc'].mean():.1f} | All≥10: {(df['wc']>=10).all()} ✅")
df.head()

Total: 4004 | Min words: 10 | Avg: 273.3 | All≥10: True ✅


,title,description,image_url,room_type,wc
0,Chic Farmhouse,Bathroom - country white tile white floor bath...,https://st.hzcdn.com/fimgs/7bf170c00b720446_29...,bathroom,28
1,Primary Bathroom Retreat,We transformed a mid-terrace Victorian bathroo...,https://st.hzcdn.com/fimgs/cfb1733d07c75ed7_91...,bathroom,57
2,Penn Valley Pearl,Inspiration for a transitional multicolored ti...,https://st.hzcdn.com/fimgs/5f9145a8078a7701_53...,bathroom,35
3,Monitor's Rest,"Monitor's Rest | Park City, Utah | CLB Archite...",https://st.hzcdn.com/fimgs/9b9138d60374162e_16...,bathroom,20
4,Aloha + Ohayo House,Bathroom Ideas A bathroom remodel can make a h...,https://st.hzcdn.com/fimgs/47a1bcd9096953e9_04...,bathroom,851


In [10]:
os.makedirs("images/bathroom", exist_ok=True)
H = {"User-Agent":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124.0.0.0 Safari/537.36"}
lp, fail = [], 0
for i, row in df.iterrows():
    fname = "images/bathroom/bathroom_" + str(i).zfill(5) + ".jpg"
    try:
        r = requests.get(row["image_url"], headers=H, timeout=15)
        if r.status_code == 200:
            Image.open(BytesIO(r.content)).convert("RGB").save(fname)
            lp.append(fname)
        else: lp.append(None); fail+=1
    except: lp.append(None); fail+=1
    time.sleep(0.4)
df["local_path"] = lp
print(f"Downloaded: {len(df)-fail} | Failed: {fail}")

Downloaded: 4004 | Failed: 0


In [11]:
df = df.dropna(subset=["local_path"]).drop(columns=["wc"],errors="ignore").reset_index(drop=True)
print(f"Final: {len(df)} images")
df.head()

Final: 4004 images


,title,description,image_url,room_type,local_path
0,Chic Farmhouse,Bathroom - country white tile white floor bath...,https://st.hzcdn.com/fimgs/7bf170c00b720446_29...,bathroom,images/bathroom/bathroom_00000.jpg
1,Primary Bathroom Retreat,We transformed a mid-terrace Victorian bathroo...,https://st.hzcdn.com/fimgs/cfb1733d07c75ed7_91...,bathroom,images/bathroom/bathroom_00001.jpg
2,Penn Valley Pearl,Inspiration for a transitional multicolored ti...,https://st.hzcdn.com/fimgs/5f9145a8078a7701_53...,bathroom,images/bathroom/bathroom_00002.jpg
3,Monitor's Rest,"Monitor's Rest | Park City, Utah | CLB Archite...",https://st.hzcdn.com/fimgs/9b9138d60374162e_16...,bathroom,images/bathroom/bathroom_00003.jpg
4,Aloha + Ohayo House,Bathroom Ideas A bathroom remodel can make a h...,https://st.hzcdn.com/fimgs/47a1bcd9096953e9_04...,bathroom,images/bathroom/bathroom_00004.jpg


In [12]:
df.to_csv("bathroom_dataset.csv", index=False)
print("Saved → bathroom_dataset.csv ✅")

Saved → bathroom_dataset.csv ✅


In [13]:
pd.read_csv("bathroom_dataset.csv")

,title,description,image_url,room_type,local_path
0,Chic Farmhouse,Bathroom - country white tile white floor bath...,https://st.hzcdn.com/fimgs/7bf170c00b720446_29...,bathroom,images/bathroom/bathroom_00000.jpg
1,Primary Bathroom Retreat,We transformed a mid-terrace Victorian bathroo...,https://st.hzcdn.com/fimgs/cfb1733d07c75ed7_91...,bathroom,images/bathroom/bathroom_00001.jpg
2,Penn Valley Pearl,Inspiration for a transitional multicolored ti...,https://st.hzcdn.com/fimgs/5f9145a8078a7701_53...,bathroom,images/bathroom/bathroom_00002.jpg
3,Monitor's Rest,"Monitor's Rest | Park City, Utah | CLB Archite...",https://st.hzcdn.com/fimgs/9b9138d60374162e_16...,bathroom,images/bathroom/bathroom_00003.jpg
4,Aloha + Ohayo House,Bathroom Ideas A bathroom remodel can make a h...,https://st.hzcdn.com/fimgs/47a1bcd9096953e9_04...,bathroom,images/bathroom/bathroom_00004.jpg
...,...,...,...,...,...
3999,Island Overlook,Freestanding bathtub - coastal gray floor free...,https://st.hzcdn.com/fimgs/ea91fc510d924b9d_45...,bathroom,images/bathroom/bathroom_03999.jpg
4000,Small Bathroom Remodel,A small bathroom remodel with Ikea vanity and ...,https://st.hzcdn.com/fimgs/dad14bba0c477ad1_43...,bathroom,images/bathroom/bathroom_04000.jpg
4001,Sonoma Master Bath,© Willett Photography www.willettphoto.com — i...,https://st.hzcdn.com/fimgs/6731815203909604_51...,bathroom,images/bathroom/bathroom_04001.jpg
4002,"MidCentury Modern ""ADA Accessible"" Guest House",Jim Bartsch Photography Example of a mid-sized...,https://st.hzcdn.com/fimgs/4ad15137041a1a52_01...,bathroom,images/bathroom/bathroom_04002.jpg
